# 🔧 Decode Your Check-Engine Light with MechanicDB

Your car's check-engine light stores a **Diagnostic Trouble Code (DTC)** — a 5-character code like `P0420` that any $20 OBD-II reader can pull. This notebook shows how **MechanicDB** turns that code into an actionable repair plan: what it means, the most likely fixes ranked by probability, whether you can do them yourself, and what the parts should cost.

This free sample contains **90 curated codes** (75 SAE-standard + 15 manufacturer-specific) in the identical schema as the full commercial dataset.

- 📦 Sample repo & docs: https://github.com/MechanicDB/MechanicDB-public
- 🌐 Full dataset (self-serve): https://mechanicdb-public.pages.dev/

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Locate the dataset wherever Kaggle mounted it (works locally too)
BASE = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")
DS = next(BASE.glob("**/dtc_codes.csv")).parent
LOAD = dict(sep="|", encoding="utf-8-sig")
read = lambda name: pd.read_csv(DS / name, **LOAD)

codes  = read("dtc_codes.csv")
joined = read("dtc_fixes_joined.csv")
parts  = read("replacement_parts.csv")

print(f"{len(codes)} codes · {len(joined)} ranked fixes · {len(parts)} parts mappings")
codes.head(3)

## 1 · Decode a code

`P0420` is one of the most-searched codes in the world — the classic "catalytic converter" light.

In [ ]:
def decode(dtc, make=None):
    q = codes[codes.dtc_code == dtc]
    if make is not None:
        q = q[q.oem_make == make]
    row = q.iloc[0]
    scope = f"{row.oem_make} (manufacturer-specific)" if row.is_oem_specific else "SAE universal"
    print(f"{row.dtc_code} — {row.short_description}")
    print(f"system: {row.system_category} · scope: {scope} · family: {row.fault_family}\n")
    print(row.detailed_technical_explanation)

decode("P0420")

## 2 · The ranked repair plan

Every code maps to 3–5 repair procedures ranked by probability (`1` = most common root cause), each with a DIY difficulty rating, a parts-cost range in USD, and estimated labor hours.

In [ ]:
def repair_plan(dtc, make=None):
    q = joined[joined.dtc_code == dtc]
    if make is not None:
        q = q[q.oem_make == make]
    cols = ["probability_rank", "fix_title", "difficulty_level",
            "est_parts_cost_min_usd", "est_parts_cost_max_usd", "est_labor_hours"]
    return q.sort_values("probability_rank")[cols].reset_index(drop=True)

repair_plan("P0420")

In [ ]:
# ...and the step-by-step instructions for the most likely fix:
top = joined[joined.dtc_code == "P0420"].sort_values("probability_rank").iloc[0]
for step in top.step_by_step_instructions.split(" • "):
    print(" ", step)

## 3 · Can I fix it myself?

The `difficulty_level` field is a three-tier rating. Let's see how DIY-able the sampled fixes are, per vehicle system.

In [ ]:
order = ["Easy DIY", "Moderate DIY", "Professional Required"]
mix = (joined.groupby(["system_category", "difficulty_level"]).size()
             .unstack(fill_value=0).reindex(columns=order))
ax = mix.plot(kind="barh", stacked=True, figsize=(9, 3.5),
              color=["#4ade80", "#f59e0b", "#ef4444"])
ax.set_title("Fix difficulty mix by vehicle system (90-code sample)")
ax.set_xlabel("ranked fixes")
plt.tight_layout(); plt.show()

In [ ]:
# Cheapest likely fix per code — a "start here" table
first = joined[joined.probability_rank == 1]
budget = (first[["dtc_code", "short_description", "fix_title",
                 "est_parts_cost_min_usd", "difficulty_level"]]
          .sort_values("est_parts_cost_min_usd").head(10))
budget

## 4 · Manufacturer-specific (OEM) codes

Beyond the universal SAE ranges, each manufacturer defines its own codes (`P1xxx` and friends) — the same number can mean **different things on different makes**. The sample includes 15 OEM rows; the full dataset covers **32 makes**.

In [ ]:
oem = codes[codes.is_oem_specific == 1]
oem[["oem_make", "dtc_code", "short_description", "fault_family"]]

In [ ]:
# The same decoder works make-aware:
decode(oem.iloc[0].dtc_code, make=oem.iloc[0].oem_make)

## 5 · Parts, joined all the way down

Each ranked fix carries 1–3 aftermarket parts with ready-made catalog search queries — useful for price lookups or affiliate integrations.

In [ ]:
fixes = read("diagnostic_fixes.csv")
(parts.merge(fixes[["fix_id", "fix_title"]], on="fix_id")
      .head(8)[["fix_title", "part_name", "amazon_search_query"]])

---

## Get the full dataset

This sample is 90 codes. The full **MechanicDB** covers:

| | Full dataset |
| :--- | ---: |
| DTC codes | 15,886 (9,249 SAE + 6,637 OEM across 32 makes) |
| Ranked repair procedures | 56,561 |
| Parts mappings | 75,055 |
| Formats | CSV · Parquet · SQLite |

Self-serve licensing with instant download: **https://mechanicdb-public.pages.dev/**

*Sample data: ODbL v1.0. Repair steps are educational reference material — high-voltage (hybrid/EV) and SRS/airbag work belongs with qualified technicians.*